# 🧠 The Policy Abstraction

Here's the central idea of `strands-robots`: **your robot code should never
change when you swap AI models.**

Whether you're using NVIDIA GR00T, a HuggingFace ACT model, or a simple test
policy, the interface is always the same:

```
observation → Policy → actions
```

This notebook shows you how that works in practice.

## What Is a Policy?

A **policy** takes in what the robot sees (camera images + joint positions) and
outputs what the robot should do (motor commands). Every policy implements the
same contract:

```python
class Policy(ABC):
    async def get_actions(self, observation_dict, instruction, **kwargs) -> list[dict]
    def get_actions_sync(self, observation_dict, instruction, **kwargs) -> list[dict]
    def set_robot_state_keys(self, robot_state_keys: list[str]) -> None
    def provider_name(self) -> str
```

**Key design choices:**
- `get_actions()` is async — real robots run event loops
- `get_actions_sync()` auto-detects if you're in an async context and does the right thing
- Observations are plain dicts — `{"camera_front": image_array, "shoulder": 0.5}`
- Actions are lists of dicts — each dict is one timestep of motor commands

## Your First Policy

Let's start with `MockPolicy` — it generates smooth sinusoidal trajectories.
No GPU, no model weights, no network. Perfect for testing.

In [ ]:
from strands_robots.policies import MockPolicy

# Create a mock policy
policy = MockPolicy()
print(f"This is a '{policy.provider_name}' policy")

# Tell it what joints the robot has
policy.set_robot_state_keys(["shoulder", "elbow", "wrist", "gripper"])

In [ ]:
# Ask it for actions — just like you would with a real AI model
observation = {"shoulder": 0.0, "elbow": 0.5, "wrist": -0.3, "gripper": 0.0}
actions = policy.get_actions_sync(observation, "pick up the red cube")

print(f"Got {len(actions)} timesteps of actions (this is called a 'chunk')")
print(f"Each timestep has keys: {list(actions[0].keys())}")
print()

# The values are smooth sinusoidal curves — different frequency per joint
for i, a in enumerate(actions):
    vals = " ".join(f"{v:+.3f}" for v in a.values())
    print(f"  t={i}: [{vals}]")

In [ ]:
# Each call advances the internal counter — you get different actions each time
# (This simulates a real policy that responds to changing observations)
actions_2 = policy.get_actions_sync(observation, "pick up the red cube")
print(f"Same actions? {actions[0] == actions_2[0]}")  # False

## The Policy Factory

In practice, you don't create policy classes directly. The `create_policy()`
factory resolves the right class from a name or even a URL:

```python
create_policy("mock")                           # → MockPolicy
create_policy("groot", port=5555)               # → Gr00tPolicy (ZMQ)
create_policy("lerobot/act_aloha_sim")          # → LerobotLocalPolicy (auto-detected)
create_policy("zmq://10.0.0.5:5555")            # → Gr00tPolicy (URL pattern)
```

In [ ]:
from strands_robots.policies import create_policy, list_providers

# See what's available
print("Available providers:", list_providers())

# Create by name
policy = create_policy("mock")
print(f"\nCreated: {policy.provider_name}")

## Building Your Own Policy

Implementing the `Policy` ABC is straightforward. Here's a minimal example:

In [ ]:
import numpy as np
from strands_robots.policies import Policy, register_policy

class RandomPolicy(Policy):
    """Uniform random actions — the simplest possible 'brain'."""

    def __init__(self, scale: float = 0.1, **kwargs):
        self._keys: list[str] = []
        self.scale = scale

    @property
    def provider_name(self) -> str:
        return "random_walk"

    def set_robot_state_keys(self, keys: list[str]) -> None:
        self._keys = keys

    async def get_actions(self, observation_dict, instruction, **kwargs):
        # Return 8 timesteps of random joint commands
        return [
            {k: float(np.random.uniform(-self.scale, self.scale)) for k in self._keys}
            for _ in range(8)
        ]

# Register it so create_policy("random_walk") works everywhere
register_policy("random_walk", lambda: RandomPolicy, aliases=["rw"])

# Test it
rw = create_policy("rw", scale=0.05)
rw.set_robot_state_keys(["shoulder", "elbow", "gripper"])

actions = rw.get_actions_sync({}, "do something random")
print("Random walk actions:")
for i, a in enumerate(actions[:3]):
    print(f"  t={i}: {a}")
print("  ...")

## Why This Matters

With this abstraction, switching from a test policy to a real AI model is a
one-line change:

```python
# Testing in simulation
policy = create_policy("mock")

# Switch to NVIDIA GR00T
policy = create_policy("groot", port=5555, data_config="so100_dualcam")

# Switch to a HuggingFace model
policy = create_policy("lerobot/act_aloha_sim_transfer_cube_human")
```

Your robot control loop stays exactly the same. That's the point.

**Next:** Notebook 03 dives into the GR00T policy — how it maps robot sensors
to model inputs, and how the ZMQ client works.